In [ ]:
!pip install -q xgboost scikit-learn pandas matplotlib numpy tensorflow

from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

# 1. GENEL AYARLAR

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

CACHE_PATH = "/content/drive/MyDrive/SIU_SLEEP/processed_dataset_final.npz"
RESULT_DIR = "/content/drive/MyDrive/SIU_SLEEP_BALANCED_FINAL_RESULTS"
os.makedirs(RESULT_DIR, exist_ok=True)

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
LABEL_TO_NAME = {0: "W", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}
N_CLASSES = 5

TARGET_CHANNELS = [
    "EEG Fpz-Cz",
    "EEG Pz-Oz",
    "EOG horizontal",
    "EMG submental"
]

BATCH_SIZE = 128
EPOCHS = 40
LEARNING_RATE = 2e-4

CLASS_WEIGHT = {
    0: 0.95,  # W
    1: 1.18,  # N1
    2: 1.00,  # N2
    3: 1.13,  # N3
    4: 1.20   # REM
}

RUN_ML_MODELS = True
RUN_MLP = True
RUN_CNN_1D = True
RUN_BILSTM = True
RUN_CNN_BILSTM = True
RUN_ADAPTIVE_FUSION = True

# 2. NPZ VERİSİNİ YÜKLE

if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(
        f"Dosya bulunamadı: {CACHE_PATH}\n"
        "CACHE_PATH yolunu kontrol et."
    )

cache = np.load(CACHE_PATH, allow_pickle=True)

X_feat_seq = cache["X_feat_seq"].astype(np.float32)
X_feat_center = cache["X_feat_center"].astype(np.float32)
X_raw_center = cache["X_raw_center"].astype(np.float32)
y_seq = cache["y_seq"].astype(np.int64)
subject_seq = cache["subject_seq"]

if "feature_names" in cache:
    feature_names = cache["feature_names"]
else:
    feature_names = np.array([f"feature_{i}" for i in range(X_feat_center.shape[1])])

print("=" * 80)
print("Veri yüklendi.")
print("X_feat_seq:", X_feat_seq.shape)
print("X_feat_center:", X_feat_center.shape)
print("X_raw_center:", X_raw_center.shape)
print("y_seq:", y_seq.shape)
print("subject_seq:", subject_seq.shape)
print("=" * 80)

print("\nGenel sınıf dağılımı:")
print(pd.Series(y_seq).map(LABEL_TO_NAME).value_counts())

# 3. SUBJECT-WISE SPLIT

unique_subjects = np.unique(subject_seq)

train_subjects, temp_subjects = train_test_split(
    unique_subjects,
    test_size=0.30,
    random_state=RANDOM_STATE
)

val_subjects, test_subjects = train_test_split(
    temp_subjects,
    test_size=2/3,
    random_state=RANDOM_STATE
)

train_mask = np.isin(subject_seq, train_subjects)
val_mask = np.isin(subject_seq, val_subjects)
test_mask = np.isin(subject_seq, test_subjects)

Xfs_train = X_feat_seq[train_mask]
Xfs_val = X_feat_seq[val_mask]
Xfs_test = X_feat_seq[test_mask]

Xfc_train = X_feat_center[train_mask]
Xfc_val = X_feat_center[val_mask]
Xfc_test = X_feat_center[test_mask]

Xraw_train = X_raw_center[train_mask]
Xraw_val = X_raw_center[val_mask]
Xraw_test = X_raw_center[test_mask]

y_train = y_seq[train_mask]
y_val = y_seq[val_mask]
y_test = y_seq[test_mask]

print("\nSubject-wise split tamamlandı.")
print("Train:", Xraw_train.shape, Xfs_train.shape, y_train.shape)
print("Validation:", Xraw_val.shape, Xfs_val.shape, y_val.shape)
print("Test:", Xraw_test.shape, Xfs_test.shape, y_test.shape)

split_dist_df = pd.DataFrame({
    "Train": pd.Series(y_train).map(LABEL_TO_NAME).value_counts(),
    "Validation": pd.Series(y_val).map(LABEL_TO_NAME).value_counts(),
    "Test": pd.Series(y_test).map(LABEL_TO_NAME).value_counts()
}).fillna(0).astype(int)

print("\nTrain / Validation / Test sınıf dağılımı:")
print(split_dist_df)

split_dist_df.to_csv(
    os.path.join(RESULT_DIR, "train_validation_test_sinif_dagilimi.csv"),
    encoding="utf-8-sig"
)

# 4. FEATURE SCALING

n_features = Xfc_train.shape[1]

scaler_center = StandardScaler()
Xfc_train_scaled = scaler_center.fit_transform(Xfc_train)
Xfc_val_scaled = scaler_center.transform(Xfc_val)
Xfc_test_scaled = scaler_center.transform(Xfc_test)

scaler_seq = StandardScaler()

Xfs_train_2d = Xfs_train.reshape(-1, n_features)
Xfs_val_2d = Xfs_val.reshape(-1, n_features)
Xfs_test_2d = Xfs_test.reshape(-1, n_features)

Xfs_train_scaled = scaler_seq.fit_transform(Xfs_train_2d).reshape(Xfs_train.shape)
Xfs_val_scaled = scaler_seq.transform(Xfs_val_2d).reshape(Xfs_val.shape)
Xfs_test_scaled = scaler_seq.transform(Xfs_test_2d).reshape(Xfs_test.shape)

X_ml_trainval = np.vstack([Xfc_train_scaled, Xfc_val_scaled])
y_ml_trainval = np.concatenate([y_train, y_val])

sample_weight_ml = np.array([CLASS_WEIGHT[int(y)] for y in y_ml_trainval])

print("\nKullanılan class weight:")
for k, v in CLASS_WEIGHT.items():
    print(f"{LABEL_TO_NAME[k]}: {v}")

# 5. METRİK VE GRAFİK FONKSİYONLARI

def evaluate_predictions(y_true, y_pred, model_name, group_name):
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    return {
        "Grup": group_name,
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Kappa": cohen_kappa_score(y_true, y_pred),
        "W F1": report["W"]["f1-score"],
        "N1 Precision": report["N1"]["precision"],
        "N1 Recall": report["N1"]["recall"],
        "N1 F1": report["N1"]["f1-score"],
        "N2 Precision": report["N2"]["precision"],
        "N2 Recall": report["N2"]["recall"],
        "N2 F1": report["N2"]["f1-score"],
        "N3 Precision": report["N3"]["precision"],
        "N3 Recall": report["N3"]["recall"],
        "N3 F1": report["N3"]["f1-score"],
        "REM F1": report["REM"]["f1-score"]
    }


def save_classification_report(y_true, y_pred, model_name):
    text = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=CLASS_NAMES,
        zero_division=0
    )

    path = os.path.join(RESULT_DIR, f"{model_name}_classification_report.txt")

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)
    print(text)


def plot_confusion_matrix_tr(y_true, y_pred, model_name):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        normalize="true"
    )

    fig, ax = plt.subplots(figsize=(7, 6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_NAMES
    )

    disp.plot(
        ax=ax,
        cmap="Blues",
        values_format=".2f",
        colorbar=True
    )

    ax.set_title(f"{model_name} Normalize Karışıklık Matrisi", fontsize=13)
    ax.set_xlabel("Tahmin Edilen Etiket", fontsize=12)
    ax.set_ylabel("Gerçek Etiket", fontsize=12)
    ax.images[-1].colorbar.set_label("Oran", fontsize=11)

    plt.tight_layout()

    path = os.path.join(RESULT_DIR, f"{model_name}_normalize_karisiklik_matrisi.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_history(history, model_name):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(history.history["accuracy"], label="Eğitim Doğruluğu")
    ax.plot(history.history["val_accuracy"], label="Doğrulama Doğruluğu")
    ax.set_xlabel("Epok")
    ax.set_ylabel("Doğruluk")
    ax.set_title(f"{model_name} Eğitim ve Doğrulama Doğruluğu")
    ax.legend()
    plt.tight_layout()
    path = os.path.join(RESULT_DIR, f"{model_name}_dogruluk_egri.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(history.history["loss"], label="Eğitim Kaybı")
    ax.plot(history.history["val_loss"], label="Doğrulama Kaybı")
    ax.set_xlabel("Epok")
    ax.set_ylabel("Kayıp")
    ax.set_title(f"{model_name} Eğitim ve Doğrulama Kaybı")
    ax.legend()
    plt.tight_layout()
    path = os.path.join(RESULT_DIR, f"{model_name}_kayip_egri.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_metric_bar(df, metric, filename, title):
    fig, ax = plt.subplots(figsize=(12, 5.5))
    ax.bar(df["Model"], df[metric])
    ax.set_xlabel("Model")
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    path = os.path.join(RESULT_DIR, filename)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_all_metrics(df):
    metrics = ["Accuracy", "Macro F1", "Kappa", "N1 F1", "N2 F1"]
    x = np.arange(len(df["Model"]))
    width = 0.15

    fig, ax = plt.subplots(figsize=(14, 6))

    for i, metric in enumerate(metrics):
        ax.bar(x + i * width, df[metric], width, label=metric)

    ax.set_xlabel("Model")
    ax.set_ylabel("Metrik Değeri")
    ax.set_title("Tüm Modellerin Dengeli Karşılaştırılması")
    ax.set_xticks(x + width * (len(metrics) - 1) / 2)
    ax.set_xticklabels(df["Model"], rotation=25, ha="right")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()

    path = os.path.join(RESULT_DIR, "tum_modeller_tum_metrikler_karsilastirma.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


def clear_memory():
    tf.keras.backend.clear_session()
    gc.collect()

# 6. DENGELİ CALLBACK

class BalancedF1Callback(tf.keras.callbacks.Callback):
    """
    En iyi epoch'u sadece val_accuracy ile değil,
    Macro F1 + N2 F1 + N1 F1 + REM F1 dengesine göre seçer.

    N2'nin çok düşmesini engellemek için N2'ye ağırlık verildi.
    """

    def __init__(self, val_data, y_val, model_name, patience=8):
        super().__init__()
        self.val_data = val_data
        self.y_val = y_val
        self.model_name = model_name
        self.patience = patience
        self.best_score = -np.inf
        self.wait = 0
        self.best_path = os.path.join(RESULT_DIR, f"{model_name}_best_balanced.weights.h5")

    def on_epoch_end(self, epoch, logs=None):
        y_prob = self.model.predict(self.val_data, batch_size=BATCH_SIZE, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        macro_f1 = f1_score(self.y_val, y_pred, average="macro", zero_division=0)

        f1_each = f1_score(
            self.y_val,
            y_pred,
            labels=[0, 1, 2, 3, 4],
            average=None,
            zero_division=0
        )

        w_f1, n1_f1, n2_f1, n3_f1, rem_f1 = f1_each

        # N2 ve REM'i korurken N1'i de tamamen bırakmayan skor
        balanced_score = (
            0.45 * macro_f1 +
            0.25 * n2_f1 +
            0.15 * n1_f1 +
            0.10 * rem_f1 +
            0.05 * n3_f1
        )

        print(
            f"\nVal MacroF1={macro_f1:.4f} | "
            f"N1F1={n1_f1:.4f} | N2F1={n2_f1:.4f} | "
            f"N3F1={n3_f1:.4f} | REMF1={rem_f1:.4f} | "
            f"BalancedScore={balanced_score:.4f}"
        )

        if balanced_score > self.best_score:
            self.best_score = balanced_score
            self.wait = 0
            self.model.save_weights(self.best_path)
            print(f"Yeni en iyi ağırlık kaydedildi: {self.best_score:.4f}")
        else:
            self.wait += 1
            print(f"İyileşme yok: {self.wait}/{self.patience}")

            if self.wait >= self.patience:
                print("Balanced callback early stopping.")
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if os.path.exists(self.best_path):
            self.model.load_weights(self.best_path)
            print(f"En iyi dengeli ağırlıklar geri yüklendi: {self.best_score:.4f}")


def get_callbacks(model_name, val_data):
    return [
        BalancedF1Callback(
            val_data=val_data,
            y_val=y_val,
            model_name=model_name,
            patience=8
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]


# 7. MODEL MİMARİLERİ

def temporal_attention(x, name_prefix="temporal"):
    score = layers.Dense(1, name=f"{name_prefix}_score")(x)
    weights = layers.Softmax(axis=1, name=f"{name_prefix}_weights")(score)
    weighted = layers.Multiply(name=f"{name_prefix}_multiply")([x, weights])

    context = layers.Lambda(
        lambda z: tf.reduce_sum(z, axis=1),
        name=f"{name_prefix}_context"
    )(weighted)

    return context


def conv_branch_raw_balanced(raw_inp, name_prefix="cnn"):
    x = layers.Conv1D(
        32, 9,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_conv1"
    )(raw_inp)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn1")(x)
    x = layers.MaxPooling1D(2, name=f"{name_prefix}_pool1")(x)
    x = layers.Dropout(0.15, name=f"{name_prefix}_drop1")(x)

    x = layers.Conv1D(
        64, 7,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_conv2"
    )(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn2")(x)
    x = layers.MaxPooling1D(2, name=f"{name_prefix}_pool2")(x)
    x = layers.Dropout(0.20, name=f"{name_prefix}_drop2")(x)

    x = layers.Conv1D(
        96, 5,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_conv3"
    )(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn3")(x)
    x = layers.GlobalAveragePooling1D(name=f"{name_prefix}_gap")(x)

    x = layers.Dense(
        96,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_dense"
    )(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_dense_bn")(x)
    x = layers.Dropout(0.35, name=f"{name_prefix}_dense_drop")(x)

    return x


def build_mlp(input_dim):
    inp = layers.Input(shape=(input_dim,))

    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)

    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.20)(x)

    out = layers.Dense(N_CLASSES, activation="softmax")(x)

    return models.Model(inp, out, name="MLP")


def build_cnn_1d(raw_shape):
    raw_inp = layers.Input(shape=raw_shape, name="raw_epoch_input")
    x = conv_branch_raw_balanced(raw_inp, name_prefix="cnn1d")
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return models.Model(raw_inp, out, name="CNN_1D")


def build_bilstm(seq_shape):
    seq_inp = layers.Input(shape=seq_shape, name="feature_sequence_input")

    x = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=True,
            dropout=0.25,
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )(seq_inp)

    x = layers.Bidirectional(
        layers.LSTM(
            32,
            return_sequences=True,
            dropout=0.25,
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )(x)

    x = temporal_attention(x, name_prefix="bilstm_attention")

    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.30)(x)

    out = layers.Dense(N_CLASSES, activation="softmax")(x)

    return models.Model(seq_inp, out, name="BiLSTM_Attention")


def build_balanced_cnn_bilstm(raw_shape, seq_shape):
    raw_inp = layers.Input(shape=raw_shape, name="raw_epoch_input")
    seq_inp = layers.Input(shape=seq_shape, name="feature_sequence_input")

    x_raw = conv_branch_raw_balanced(raw_inp, name_prefix="balanced_cnn_bilstm_raw")

    x_seq = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=True,
            dropout=0.25,
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )(seq_inp)

    x_seq = layers.Bidirectional(
        layers.LSTM(
            32,
            return_sequences=True,
            dropout=0.25,
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )(x_seq)

    x_seq = temporal_attention(x_seq, name_prefix="balanced_cnn_bilstm_attention")

    x = layers.Concatenate(name="balanced_cnn_bilstm_fusion")([x_raw, x_seq])

    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.25)(x)

    out = layers.Dense(N_CLASSES, activation="softmax")(x)

    return models.Model([raw_inp, seq_inp], out, name="Balanced_CNN_BiLSTM")


def build_balanced_adaptive_fusion(raw_shape, seq_shape, feature_dim):
    raw_inp = layers.Input(shape=raw_shape, name="raw_epoch_input")
    seq_inp = layers.Input(shape=seq_shape, name="feature_sequence_input")
    feat_inp = layers.Input(shape=(feature_dim,), name="center_feature_input")

    n_channels = raw_shape[-1]
    channel_embeddings = []

    for ch in range(n_channels):
        ch_signal = layers.Lambda(
            lambda z, c=ch: z[:, :, c:c+1],
            name=f"channel_{ch}_slice"
        )(raw_inp)

        x = layers.Conv1D(
            16, 9,
            padding="same",
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        )(ch_signal)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Dropout(0.15)(x)

        x = layers.Conv1D(
            32, 7,
            padding="same",
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Dropout(0.20)(x)

        x = layers.Conv1D(
            64, 5,
            padding="same",
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling1D()(x)

        x = layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4)
        )(x)
        x = layers.Dropout(0.25)(x)

        channel_embeddings.append(x)

    stacked = layers.Lambda(
        lambda tensors: tf.stack(tensors, axis=1),
        name="channel_embedding_stack"
    )(channel_embeddings)

    channel_scores = layers.Dense(1, name="adaptive_channel_score")(stacked)
    channel_weights = layers.Softmax(axis=1, name="adaptive_channel_weights")(channel_scores)

    weighted_channels = layers.Multiply(name="adaptive_channel_multiply")(
        [stacked, channel_weights]
    )

    fused_channel = layers.Lambda(
        lambda z: tf.reduce_sum(z, axis=1),
        name="adaptive_channel_fusion"
    )(weighted_channels)

    fused_channel = layers.BatchNormalization()(fused_channel)
    fused_channel = layers.Dropout(0.30)(fused_channel)

    x_seq = layers.Bidirectional(
        layers.LSTM(
            64,
            return_sequences=True,
            dropout=0.25,
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )(seq_inp)

    x_seq = layers.Bidirectional(
        layers.LSTM(
            32,
            return_sequences=True,
            dropout=0.25,
            kernel_regularizer=regularizers.l2(1e-4)
        )
    )(x_seq)

    x_seq = temporal_attention(x_seq, name_prefix="adaptive_temporal_attention")

    x_seq = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x_seq)
    x_seq = layers.BatchNormalization()(x_seq)
    x_seq = layers.Dropout(0.30)(x_seq)

    x_feat = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(feat_inp)
    x_feat = layers.BatchNormalization()(x_feat)
    x_feat = layers.Dropout(0.30)(x_feat)

    x_feat = layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x_feat)
    x_feat = layers.Dropout(0.20)(x_feat)

    x = layers.Concatenate(name="balanced_adaptive_fusion_concat")(
        [fused_channel, x_seq, x_feat]
    )

    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.45)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.25)(x)

    out = layers.Dense(N_CLASSES, activation="softmax")(x)

    return models.Model(
        [raw_inp, seq_inp, feat_inp],
        out,
        name="Balanced_AdaptiveFusion"
    )

# 8. DL TRAIN FUNCTION

def train_and_evaluate_dl(
    model,
    train_data,
    val_data,
    test_data,
    model_name,
    use_class_weight=True
):
    print("\n" + "=" * 80)
    print(f"{model_name} eğitiliyor...")
    print("=" * 80)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()

    callbacks = get_callbacks(model_name, val_data)

    fit_kwargs = {
        "x": train_data,
        "y": y_train,
        "validation_data": (val_data, y_val),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "callbacks": callbacks,
        "verbose": 1
    }

    if use_class_weight:
        fit_kwargs["class_weight"] = CLASS_WEIGHT

    history = model.fit(**fit_kwargs)

    y_prob = model.predict(test_data, batch_size=BATCH_SIZE)
    y_pred = np.argmax(y_prob, axis=1)

    result = evaluate_predictions(y_test, y_pred, model_name, "Deep Learning")

    save_classification_report(y_test, y_pred, model_name)
    plot_history(history, model_name)
    plot_confusion_matrix_tr(y_test, y_pred, model_name)

    return result, y_pred, y_prob, history, model


# 9. MODEL EĞİTİMLERİ

all_results = []
all_predictions = {}

# -------------------------
# ML MODELLERİ
# -------------------------

if RUN_ML_MODELS:
    print("\n" + "=" * 80)
    print("MAKİNE ÖĞRENMESİ MODELLERİ")
    print("=" * 80)

    rf_model = RandomForestClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        class_weight=CLASS_WEIGHT,
        n_jobs=-1
    )

    rf_model.fit(X_ml_trainval, y_ml_trainval)
    rf_pred = rf_model.predict(Xfc_test_scaled)

    all_results.append(evaluate_predictions(y_test, rf_pred, "Random Forest", "Makine Öğrenmesi"))
    all_predictions["Random Forest"] = rf_pred
    save_classification_report(y_test, rf_pred, "Random_Forest")
    plot_confusion_matrix_tr(y_test, rf_pred, "Random_Forest")

    del rf_model
    gc.collect()

    svm_model = SVC(
        kernel="rbf",
        C=3.0,
        gamma="scale",
        class_weight=CLASS_WEIGHT,
        probability=False,
        random_state=RANDOM_STATE
    )

    svm_model.fit(X_ml_trainval, y_ml_trainval)
    svm_pred = svm_model.predict(Xfc_test_scaled)

    all_results.append(evaluate_predictions(y_test, svm_pred, "SVM", "Makine Öğrenmesi"))
    all_predictions["SVM"] = svm_pred
    save_classification_report(y_test, svm_pred, "SVM")
    plot_confusion_matrix_tr(y_test, svm_pred, "SVM")

    del svm_model
    gc.collect()

    xgb_model = XGBClassifier(
        objective="multi:softmax",
        num_class=N_CLASSES,
        n_estimators=400,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2.5,
        reg_alpha=0.3,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    xgb_model.fit(
        X_ml_trainval,
        y_ml_trainval,
        sample_weight=sample_weight_ml
    )

    xgb_pred = xgb_model.predict(Xfc_test_scaled)

    all_results.append(evaluate_predictions(y_test, xgb_pred, "XGBoost", "Makine Öğrenmesi"))
    all_predictions["XGBoost"] = xgb_pred
    save_classification_report(y_test, xgb_pred, "XGBoost")
    plot_confusion_matrix_tr(y_test, xgb_pred, "XGBoost")

    del xgb_model
    gc.collect()

# DL MODELLERİ

raw_shape = Xraw_train.shape[1:]
seq_shape = Xfs_train_scaled.shape[1:]
feature_dim = Xfc_train_scaled.shape[1]

print("\n" + "=" * 80)
print("DEEP LEARNING MODELLERİ")
print("=" * 80)

if RUN_MLP:
    mlp_model = build_mlp(feature_dim)

    res, pred, prob, hist, trained = train_and_evaluate_dl(
        model=mlp_model,
        train_data=Xfc_train_scaled,
        val_data=Xfc_val_scaled,
        test_data=Xfc_test_scaled,
        model_name="MLP",
        use_class_weight=True
    )

    all_results.append(res)
    all_predictions["MLP"] = pred

    del mlp_model, trained, hist, prob
    clear_memory()


if RUN_CNN_1D:
    cnn_model = build_cnn_1d(raw_shape)

    res, pred, prob, hist, trained = train_and_evaluate_dl(
        model=cnn_model,
        train_data=Xraw_train,
        val_data=Xraw_val,
        test_data=Xraw_test,
        model_name="CNN_1D",
        use_class_weight=True
    )

    all_results.append(res)
    all_predictions["CNN_1D"] = pred

    del cnn_model, trained, hist, prob
    clear_memory()


if RUN_BILSTM:
    bilstm_model = build_bilstm(seq_shape)

    res, pred, prob, hist, trained = train_and_evaluate_dl(
        model=bilstm_model,
        train_data=Xfs_train_scaled,
        val_data=Xfs_val_scaled,
        test_data=Xfs_test_scaled,
        model_name="BiLSTM_Attention",
        use_class_weight=True
    )

    all_results.append(res)
    all_predictions["BiLSTM_Attention"] = pred

    del bilstm_model, trained, hist, prob
    clear_memory()


if RUN_CNN_BILSTM:
    cnn_bilstm_model = build_balanced_cnn_bilstm(raw_shape, seq_shape)

    res, pred, prob, hist, trained = train_and_evaluate_dl(
        model=cnn_bilstm_model,
        train_data=[Xraw_train, Xfs_train_scaled],
        val_data=[Xraw_val, Xfs_val_scaled],
        test_data=[Xraw_test, Xfs_test_scaled],
        model_name="Balanced_CNN_BiLSTM",
        use_class_weight=True
    )

    all_results.append(res)
    all_predictions["Balanced_CNN_BiLSTM"] = pred

    del cnn_bilstm_model, trained, hist, prob
    clear_memory()


if RUN_ADAPTIVE_FUSION:
    adaptive_model = build_balanced_adaptive_fusion(
        raw_shape=raw_shape,
        seq_shape=seq_shape,
        feature_dim=feature_dim
    )

    res, pred, prob, hist, trained = train_and_evaluate_dl(
        model=adaptive_model,
        train_data=[Xraw_train, Xfs_train_scaled, Xfc_train_scaled],
        val_data=[Xraw_val, Xfs_val_scaled, Xfc_val_scaled],
        test_data=[Xraw_test, Xfs_test_scaled, Xfc_test_scaled],
        model_name="Balanced_AdaptiveFusion",
        use_class_weight=True
    )

    all_results.append(res)
    all_predictions["Balanced_AdaptiveFusion"] = pred

    # Kanal ağırlıkları
    try:
        attention_model = models.Model(
            inputs=trained.input,
            outputs=trained.get_layer("adaptive_channel_weights").output
        )

        attn = attention_model.predict(
            [Xraw_test, Xfs_test_scaled, Xfc_test_scaled],
            batch_size=BATCH_SIZE,
            verbose=0
        )

        mean_attn = np.mean(attn.squeeze(-1), axis=0)

        channel_df = pd.DataFrame({
            "Kanal": TARGET_CHANNELS,
            "Ortalama Dikkat Ağırlığı": mean_attn
        })

        channel_path = os.path.join(RESULT_DIR, "adaptive_channel_weights.csv")
        channel_df.to_csv(channel_path, index=False, encoding="utf-8-sig")

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.bar(channel_df["Kanal"], channel_df["Ortalama Dikkat Ağırlığı"])
        ax.set_xlabel("PSG Kanalı")
        ax.set_ylabel("Ortalama Dikkat Ağırlığı")
        ax.set_title("Balanced Adaptive Fusion Kanal Ağırlıkları")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()

        fig_path = os.path.join(RESULT_DIR, "adaptive_channel_weights.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("\nAdaptive channel weights:")
        print(channel_df)

    except Exception as e:
        print("Adaptive channel weights çıkarılamadı:", e)

    del adaptive_model, trained, hist, prob
    clear_memory()

# 10. SONUÇ TABLOSU VE GRAFİKLER

all_results_df = pd.DataFrame(all_results)

ordered_cols = [
    "Grup",
    "Model",
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
    "Kappa",
    "N1 Precision",
    "N1 Recall",
    "N1 F1",
    "N2 Precision",
    "N2 Recall",
    "N2 F1",
    "W F1",
    "N3 F1",
    "REM F1"
]

all_results_df = all_results_df[ordered_cols]

print("\n" + "=" * 80)
print("TÜM MODEL KARŞILAŞTIRMA SONUÇLARI")
print("=" * 80)
print(all_results_df.round(4))

results_path = os.path.join(RESULT_DIR, "tum_model_karsilastirma_sonuclari.csv")
all_results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

print("Sonuç tablosu kaydedildi:", results_path)

plot_metric_bar(
    all_results_df,
    "Accuracy",
    "accuracy_tum_modeller.png",
    "Tüm Modellerin Accuracy Karşılaştırması"
)

plot_metric_bar(
    all_results_df,
    "Macro F1",
    "macro_f1_tum_modeller.png",
    "Tüm Modellerin Macro F1 Karşılaştırması"
)

plot_metric_bar(
    all_results_df,
    "Kappa",
    "kappa_tum_modeller.png",
    "Tüm Modellerin Kappa Karşılaştırması"
)

plot_metric_bar(
    all_results_df,
    "N1 F1",
    "n1_f1_tum_modeller.png",
    "Tüm Modellerin N1 F1 Karşılaştırması"
)

plot_metric_bar(
    all_results_df,
    "N2 F1",
    "n2_f1_tum_modeller.png",
    "Tüm Modellerin N2 F1 Karşılaştırması"
)

plot_all_metrics(all_results_df)

# 11. ÖZET


best_acc = all_results_df.sort_values("Accuracy", ascending=False).iloc[0]
best_macro = all_results_df.sort_values("Macro F1", ascending=False).iloc[0]
best_kappa = all_results_df.sort_values("Kappa", ascending=False).iloc[0]
best_n1 = all_results_df.sort_values("N1 F1", ascending=False).iloc[0]
best_n2 = all_results_df.sort_values("N2 F1", ascending=False).iloc[0]

summary_text = f"""
ÖZET SONUÇLAR
==================================================

Accuracy açısından en iyi model:
{best_acc['Model']} | Accuracy = {best_acc['Accuracy']:.4f}

Macro F1 açısından en iyi model:
{best_macro['Model']} | Macro F1 = {best_macro['Macro F1']:.4f}

Kappa açısından en iyi model:
{best_kappa['Model']} | Kappa = {best_kappa['Kappa']:.4f}

N1 F1 açısından en iyi model:
{best_n1['Model']} | N1 F1 = {best_n1['N1 F1']:.4f}

N2 F1 açısından en iyi model:
{best_n2['Model']} | N2 F1 = {best_n2['N2 F1']:.4f}

Ana önerilen model adayları:
1) Balanced_CNN_BiLSTM
2) Balanced_AdaptiveFusion

Not:
Bu kodda focal loss kullanılmamıştır.
Amaç N1 ve N3'ü artırırken N2'nin aşırı düşmesini engellemektir.
En iyi epoch seçimi val_accuracy yerine Macro F1 + N2 F1 + N1 F1 dengesine göre yapılmıştır.
"""

print(summary_text)

summary_path = os.path.join(RESULT_DIR, "ozet_sonuclar.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)

print("Özet kaydedildi:", summary_path)
print("Tüm çıktılar:", RESULT_DIR)